# RecRec: Strict Paper-Adhering PyTorch Lightning Implementation
#
Paper: "RecRec: Recursive Refinement for Sequential Recommendation"
#
The written paper is the specification. The released implementation is
not silently used to fill paper ambiguities.
#
Paper-supported settings:
- d=384
- max history length=50
- T=7 outer refinement steps
- n=3 inner recursive updates
- candidate set size=100
- Adam lr=1e-3
- batch size=512
- 50 epochs
- EMA decay=0.999
#
Paper-defined architecture:
- frozen SBERT item embeddings
- masked mean history aggregation
- y0=x, z0=0
- shared f_phi across all recursive applications
- step-specific correction matrices W_t
- evidence-anchored correction gate
- residual preference update with scaling L
- deep supervision averaged across T steps
#
Explicitly not specified numerically by the paper:
- the value of L
- the exact hidden-width convention for the ablated MLP depth
- the exact negative-sampling distribution beyond sampled negatives
- a separate validation split
#
These are therefore exposed/configured rather than silently copied from the
authors' released implementation.


In [1]:
!uv pip install pytorch_lightning -q

In [2]:
from dataclasses import dataclass
from collections import defaultdict
from pathlib import Path
from typing import Optional, Sequence

import math
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback
import shutil
# shutil.copytree("/kaggle/input/datasets/chrisolande2/recsys", "data")

## 1. Reproducibility


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



## 2. Configuration
#
`core_depth=5` follows the paper's Section 5.2 result where five layers
gives the best reported depth setting.
#
`preference_scale=L` defaults to 1.0 because the paper introduces L but does
not provide a numerical value. This is explicitly a configuration choice.
#
`temperature` is likewise exposed because the paper uses tau in Eq. (4) but
does not provide its numeric value in the text supplied here.


In [4]:
@dataclass
class RecRecConfig:
    embedding_dim: int = 384
    max_history_length: int = 50
    outer_steps: int = 7
    inner_steps: int = 3
    # Best depth reported in Section 5.2
    core_depth: int = 5
    # Eq. (3), numerical value not specified by the paper
    preference_scale: float = 1.0
    # Eq. (4), numerical value not specified by the paper
    temperature: float = 1.0
    # Appendix A.1
    candidate_size: int = 100
    learning_rate: float = 1e-3
    batch_size: int = 512
    max_epochs: int = 50
    ema_decay: float = 0.999
    # Item embeddings fine-tuning
    freeze_item_embeddings: bool = False
    num_workers: int = 3
    # Paper says sampled negatives. The exact sampling law is not specified.
    # We use uniform sampling without replacement and exclude target/history.
    exclude_history_items_from_negatives: bool = True
CONFIG = RecRecConfig()


## 3. Data Loading
#
Interaction file format:
    user_id item_id
#
User interactions must already be chronological. Item IDs must be contiguous
0..N-1 and must correspond exactly to SBERT embedding rows.


In [5]:
def load_user_sequences(interaction_path: str | Path) -> dict[int, list[int]]:
    sequences: dict[int, list[int]] = defaultdict(list)

    with open(interaction_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            user_id, item_id = map(int, parts[:2])
            sequences[user_id].append(item_id)

    sequences = {
        user_id: seq
        for user_id, seq in sequences.items()
        if len(seq) >= 2
    }

    if not sequences:
        raise ValueError("No valid user sequences found.")

    return sequences


In [6]:
def validate_item_indexing(
    user_sequences: dict[int, list[int]],
    num_items: int,
) -> None:
    ids = [item_id for seq in user_sequences.values() for item_id in seq]
    if not ids:
        raise ValueError("No item IDs found.")

    if min(ids) < 0:
        raise ValueError("Negative item ID found.")

    if max(ids) >= num_items:
        raise ValueError(
            f"Item ID {max(ids)} exceeds embedding table size {num_items}."
        )



## 4. SBERT Embedding Extraction
#
Section 3/4 states that each item has a pretrained semantic embedding from
frozen SBERT embeddings of item metadata.
#
`nn.Embedding` is only the lookup table. SBERT extraction happens here.


In [7]:
def extract_sbert_item_embeddings(
    interaction_path: str | Path,
    metadata_path: str | Path,
    output_path: str | Path,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    batch_size: int = 512,
    device: str | None = None,
) -> torch.Tensor:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "gpu"

    user_sequences = load_user_sequences(interaction_path)
    item_ids = sorted({i for seq in user_sequences.values() for i in seq})

    expected_ids = list(range(len(item_ids)))
    if item_ids != expected_ids:
        raise ValueError(
            "Item IDs must be contiguous 0..N-1 before SBERT extraction."
        )

    with open(metadata_path, "rb") as f:
        metadata = pickle.load(f)

    if isinstance(metadata, dict) and "title" in metadata:
        id_to_title = metadata["title"]
    else:
        id_to_title = metadata

    texts = []
    for item_id in item_ids:
        text = id_to_title.get(item_id, "unknown item")
        if text is None or not str(text).strip():
            text = "unknown item"
        texts.append(str(text))

    sbert = SentenceTransformer(model_name).to(device)

    chunks = []
    for start in tqdm(range(0, len(texts), batch_size), desc="SBERT"):
        batch_texts = texts[start:start + batch_size]
        chunks.append(
            sbert.encode(
                batch_texts,
                convert_to_tensor=True,
                device=device,
                normalize_embeddings=True,
            )
        )

    item_embeddings = torch.cat(chunks, dim=0).to(device)

    if item_embeddings.shape != (len(item_ids), 384):
        raise RuntimeError(
            f"Expected ({len(item_ids)}, 384), got {tuple(item_embeddings.shape)}"
        )

    if not torch.isfinite(item_embeddings).all():
        raise RuntimeError("SBERT embedding matrix contains non-finite values.")

    torch.save(item_embeddings, output_path)
    return item_embeddings



## 5. Leave-One-Out Construction
#
The paper explicitly describes leave-one-out evaluation. It does not specify
the random-prefix training sampler found in the released code. Therefore the
strict paper implementation uses one leave-one-out target per user:
#
    history = sequence[:-1]
    target  = sequence[-1]
#
The same task is evaluated with a sampled candidate set.


In [8]:
def make_train_val_pairs(
    user_sequences: dict[int, list[int]],
) -> tuple[list[tuple[list[int], int]], list[tuple[list[int], int]]]:
    train_pairs = []
    val_pairs = []

    for seq in user_sequences.values():
        if len(seq) < 3:
            continue

        # Validation task: strictly leave-one-out (predict the last item)
        val_pairs.append((seq[:-1], seq[-1]))

        # Training tasks: all earlier causal prefixes
        for i in range(1, len(seq) - 1):
            train_pairs.append((seq[:i], seq[i]))

    return train_pairs, val_pairs


## 6. Candidate Sampling
#
Paper: candidate set = ground-truth item + sampled negatives.
Exact negative distribution is not specified, so uniform sampling is isolated
here as an explicit implementation choice.


In [9]:
def sample_candidate_set(
    target_item: int,
    history: Sequence[int],
    num_items: int,
    candidate_size: int,
    exclude_history: bool = True,
) -> tuple[list[int], int]:
    forbidden = {target_item}
    if exclude_history:
        forbidden.update(history)

    available = [i for i in range(num_items) if i not in forbidden]
    if len(available) < candidate_size - 1:
        raise ValueError("Not enough negatives to construct candidate set.")

    negatives = random.sample(available, candidate_size - 1)
    candidates = negatives + [target_item]
    random.shuffle(candidates)
    return candidates, candidates.index(target_item)



In [10]:
class RecRecDataset(Dataset):
    def __init__(self, pairs: Sequence[tuple[Sequence[int], int]]):
        self.pairs = [(list(h), int(t)) for h, t in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]

In [11]:
class RecRecCollator:
    def __init__(self, config: RecRecConfig, num_items: int):
        self.max_history_length = config.max_history_length
        self.candidate_size = config.candidate_size
        self.num_items = num_items
        self.exclude_history = config.exclude_history_items_from_negatives

    def __call__(self, batch):
        batch_size = len(batch)

        history_ids = torch.zeros(
            batch_size, self.max_history_length, dtype=torch.long
        )
        history_mask = torch.zeros(
            batch_size, self.max_history_length, dtype=torch.float32
        )
        candidate_ids = torch.zeros(
            batch_size, self.candidate_size, dtype=torch.long
        )
        target_index = torch.zeros(batch_size, dtype=torch.long)

        for row, (history, target) in enumerate(batch):
            history = history[-self.max_history_length:]
            length = len(history)

            history_ids[row, -length:] = torch.tensor(
                history, dtype=torch.long
            )
            history_mask[row, -length:] = 1.0

            candidates, target_position = sample_candidate_set(
                target_item=target,
                history=history,
                num_items=self.num_items,
                candidate_size=self.candidate_size,
                exclude_history=self.exclude_history,
            )

            candidate_ids[row] = torch.tensor(
                candidates, dtype=torch.long
            )
            target_index[row] = target_position

        return history_ids, history_mask, candidate_ids, target_index



In [12]:
class RecRecDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_pairs: Sequence[tuple[Sequence[int], int]],
        val_pairs: Sequence[tuple[Sequence[int], int]],
        num_items: int,
        config: RecRecConfig,
    ):
        super().__init__()
        self.train_pairs = list(train_pairs)
        self.val_pairs = list(val_pairs)
        self.num_items = num_items
        self.config = config
        self.collator = RecRecCollator(config, num_items)

    def setup(self, stage:str | None = None):
        self.train_dataset = RecRecDataset(self.train_pairs)
        self.val_dataset = RecRecDataset(self.val_pairs)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
            pin_memory=torch.cuda.is_available(),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
            pin_memory=torch.cuda.is_available(),
        )


## 7. Input Encoding
#
Paper:
#
    x = masked mean of history item embeddings
    y0 = x
    z0 = 0


In [13]:
class InputEncoding(nn.Module):
    def forward(self, item_weight, history_item_ids, history_mask):
        history_embeddings = F.embedding(history_item_ids, item_weight)
        mask = history_mask.to(history_embeddings.dtype).unsqueeze(-1)

        denominator = mask.sum(dim=1).clamp_min(1e-12)
        x = (history_embeddings * mask).sum(dim=1) / denominator

        y0 = x
        z0 = torch.zeros_like(x)
        return x, y0, z0



## 8. Shared Recursive Core f_phi
#
The paper states that a shared nonlinear transformation f_phi is applied
recursively. Section 5.2 reports that depth 5 is the best setting.
#
The paper does not fully formalize the hidden-width convention for the MLP
depth ablation. We therefore use the following transparent convention:
    first layer: 3d -> d
    subsequent layers: d -> d
with LayerNorm after each Linear and ReLU between layers.


In [14]:
class CoreRecursionMLP(nn.Module):
    def __init__(self, embedding_dim: int, depth: int):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be >= 1")

        layers = []
        for layer_idx in range(depth):
            in_dim = 3 * embedding_dim if layer_idx == 0 else embedding_dim
            layers.append(nn.Linear(in_dim, embedding_dim))
            layers.append(nn.LayerNorm(embedding_dim))
            if layer_idx < depth - 1:
                layers.append(nn.ReLU())

        self.network = nn.Sequential(*layers)

    def forward(self, state):
        return self.network(state)



## 9. Recursive Preference Refinement
#
Eq. (1): z_t^(j) = f_phi([x || y_t || z_t^(j-1)])

Eq. (2): g_t = sigmoid(W_t [x || y_t])
Correction: z_t = (1-g_t) z_t^(n) + g_t x
Eq. (3): y_(t+1) = y_t + L tanh(f_phi([x || y_t || z_t]))
#
W_t is distinct for every outer step, while f_phi is shared.


In [15]:
class RecursivePreferenceRefinement(nn.Module):
    def __init__(self, config: RecRecConfig):
        super().__init__()

        d = config.embedding_dim
        self.outer_steps = config.outer_steps
        self.inner_steps = config.inner_steps
        self.preference_scale = config.preference_scale

        self.f_phi = CoreRecursionMLP(
            embedding_dim=d,
            depth=config.core_depth,
        )

        self.correction_gates = nn.ModuleList(
            [nn.Linear(2 * d, d) for _ in range(config.outer_steps)]
        )

    def forward(self, x, y0, z0):
        y = y0
        z = z0
        y_states = []

        for t in range(self.outer_steps):
            z_inner = z

            for _ in range(self.inner_steps):
                z_inner = self.f_phi(
                    torch.cat([x, y, z_inner], dim=-1)
                )

            g = torch.sigmoid(
                self.correction_gates[t](
                    torch.cat([x, y], dim=-1)
                )
            )

            z = (1.0 - g) * z_inner + g * x

            delta = torch.tanh(
                self.f_phi(
                    torch.cat([x, y, z], dim=-1)
                )
            )

            y = y + self.preference_scale * delta
            y_states.append(y)

        return y_states



## 10. Candidate Scoring
#
Eq. (4) uses:
    y_(t+1) dot e_j / tau
#
Temperature is a configurable paper parameter because the supplied paper
text does not state its numerical value.


In [16]:
class CandidateScoring(nn.Module):
    def __init__(self, temperature: float):
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive")
        self.temperature = temperature

    def forward(self, item_weight, y_states, candidate_ids):
        candidate_embeddings = F.embedding(candidate_ids, item_weight)
        logits = []

        for y in y_states:
            scores = torch.einsum(
                "bd,bnd->bn",
                y,
                candidate_embeddings,
            )
            logits.append(scores / self.temperature)

        return logits



## 11. Complete RecRec Architecture


In [17]:
class RecRec(nn.Module):
    def __init__(self, pretrained_sbert_embeddings, config: RecRecConfig):
        super().__init__()

        if pretrained_sbert_embeddings.ndim != 2:
            raise ValueError("SBERT embeddings must be [num_items, dim]")

        if pretrained_sbert_embeddings.size(1) != config.embedding_dim:
            raise ValueError(
                f"Expected embedding dim {config.embedding_dim}, "
                f"got {pretrained_sbert_embeddings.size(1)}"
            )

        self.item_embeddings = nn.Embedding.from_pretrained(
            pretrained_sbert_embeddings.float(),
            freeze=config.freeze_item_embeddings,
        )

        self.input_encoding = InputEncoding()
        self.preference_refinement = RecursivePreferenceRefinement(config)
        self.candidate_scoring = CandidateScoring(config.temperature)

    @property
    def item_weight(self):
        return self.item_embeddings.weight

    def forward(self, history_ids, history_mask, candidate_ids=None):
        x, y0, z0 = self.input_encoding(
            self.item_weight,
            history_ids,
            history_mask,
        )

        y_states = self.preference_refinement(x, y0, z0)

        if candidate_ids is None:
            return y_states[-1]

        return self.candidate_scoring(
            self.item_weight,
            y_states,
            candidate_ids,
        )



## 12. Deep Supervision Loss
#
Paper Eq. (4):
#
    L_total = 1/T * sum_t L_CE(y_(t+1), target)
#
The implementation therefore averages the step losses.
#
This intentionally differs from the released code's summed loss.


In [18]:
def deep_supervision_loss(logits_per_step, target_index):
    return torch.stack([
        F.cross_entropy(logits, target_index)
        for logits in logits_per_step
    ]).mean()



## 13. EMA
#
The paper's Appendix specifies EMA decay 0.999.


In [19]:
class EMACallback(Callback):
    def __init__(self, decay: float = 0.999):
        super().__init__()
        self.decay = decay
        self.shadow: dict[str, torch.Tensor] = {}
        self.backup: dict[str, torch.Tensor] = {}

    def _trainable_parameters(self, module):
        return (
            (name, parameter)
            for name, parameter in module.named_parameters()
            if parameter.requires_grad
        )

    def _initialize(self, module):
        if not self.shadow:
            self.shadow = {
                name: p.detach().clone()
                for name, p in self._trainable_parameters(module)
            }

    @torch.no_grad()
    def on_train_start(self, trainer, pl_module):
        self._initialize(pl_module)

    @torch.no_grad()
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self._initialize(pl_module)

        for name, p in self._trainable_parameters(pl_module):
            if name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(
                    p.detach(), alpha=1.0 - self.decay
                )

    @torch.no_grad()
    def apply_shadow(self, pl_module):
        if not self.shadow:
            return

        self.backup = {}
        for name, p in self._trainable_parameters(pl_module):
            if name in self.shadow:
                self.backup[name] = p.detach().clone()
                p.copy_(self.shadow[name])

    @torch.no_grad()
    def restore(self, pl_module):
        if not self.backup:
            return

        for name, p in self._trainable_parameters(pl_module):
            if name in self.backup:
                p.copy_(self.backup[name])
        self.backup.clear()

    def on_validation_epoch_start(self, trainer, pl_module):
        self.apply_shadow(pl_module)

    def on_validation_epoch_end(self, trainer, pl_module):
        self.restore(pl_module)

    def on_test_epoch_start(self, trainer, pl_module):
        self.apply_shadow(pl_module)

    def on_test_epoch_end(self, trainer, pl_module):
        self.restore(pl_module)


## 14. Metrics Matching the Paper's Tables
#
HR, NDCG, and Precision are computed at k in {1,5,10}.
#
With one relevant item, precision@k is 1/k when the target is in the top k.


In [20]:
def compute_ranking_metrics(ranks: torch.Tensor) -> dict[str, float]:
    ranks = ranks.float()
    metrics = {}

    for k in (1, 5, 10):
        hit = ranks <= k
        metrics[f"HR@{k}"] = hit.float().mean().item()
        metrics[f"NDCG@{k}"] = torch.where(
            hit,
            1.0 / torch.log2(ranks + 1.0),
            torch.zeros_like(ranks),
        ).mean().item()
        metrics[f"Prec@{k}"] = (
            hit.float() / float(k)
        ).mean().item()

    return metrics



## 15. Lightning Module
#
Training uses the paper's averaged deep-supervision objective.
Validation uses the final refined state y_T and candidate ranking.


In [21]:
class RecRecLightning(pl.LightningModule):
    def __init__(self, pretrained_sbert_embeddings, config: RecRecConfig,device = None):
        super().__init__()
        self.save_hyperparameters(ignore=["pretrained_sbert_embeddings"])
        self.config = config
        self.model = RecRec(pretrained_sbert_embeddings, config)
        self._validation_ranks = []
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

    def forward(self, history_ids, history_mask, candidate_ids=None):
        return self.model(history_ids, history_mask, candidate_ids)

    def training_step(self, batch, batch_idx):
        history_ids, history_mask, candidate_ids, target_index = batch

        logits = self(
            history_ids,
            history_mask,
            candidate_ids,
        )

        loss = deep_supervision_loss(logits, target_index)

        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            batch_size=history_ids.size(0),
        )

        return loss

    def on_validation_epoch_start(self):
        self._validation_ranks = []

    @torch.no_grad()
    def validation_step(self, batch, batch_idx):
        history_ids, history_mask, candidate_ids, target_index = batch

        logits = self(
            history_ids,
            history_mask,
            candidate_ids,
        )

        final_logits = logits[-1]

        ranking = torch.argsort(
            final_logits,
            dim=1,
            descending=True,
        )

        ranks = (
            (ranking == target_index.unsqueeze(1))
            .nonzero(as_tuple=True)[1]
            + 1
        )

        self._validation_ranks.append(ranks.to(device))

    def on_validation_epoch_end(self):
        if not self._validation_ranks:
            return

        ranks = torch.cat(self._validation_ranks)
        metrics = compute_ranking_metrics(ranks)
        self._validation_ranks.clear()

        for name, value in metrics.items():
            self.log(
                f"val_{name.lower().replace('@', '')}",
                value,
                prog_bar=(name == "NDCG@10"),
            )

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.config.learning_rate,
        )



## 16. Paper-Compliance Checks
#
These checks are deliberately mechanical. They make it difficult to drift
back toward the released implementation while experimenting.


In [22]:
def validate_paper_compliance(model: RecRecLightning, config: RecRecConfig):
    assert config.embedding_dim == 384
    assert config.max_history_length == 50
    assert config.outer_steps == 7
    assert config.inner_steps == 3
    assert config.candidate_size == 100
    assert config.learning_rate == 1e-3
    assert config.batch_size == 512
    assert config.max_epochs == 50
    assert config.ema_decay == 0.999
    assert config.freeze_item_embeddings is False

    item_table = model.model.item_embeddings
    assert item_table.weight.requires_grad is True

    refinement = model.model.preference_refinement
    assert len(refinement.correction_gates) == config.outer_steps

    # Only one f_phi object exists, hence it is shared across all recursive
    # applications.
    assert isinstance(refinement.f_phi, CoreRecursionMLP)


## 17. One-Batch Numerical Diagnostic
#
For 100 candidates, a uniform predictor has cross-entropy log(100)=4.605.
This is only a sanity baseline. The purpose is to catch broken embedding
alignment, exploding logits, wrong targets, or other pipeline errors BEFORE
running 50 epochs.


In [23]:
@torch.no_grad()
def inspect_one_batch(model: RecRecLightning, loader: DataLoader):
    # Detect the actual device of the model parameters
    device = next(model.parameters()).device

    batch = next(iter(loader))
    batch = [x.to(device) for x in batch]
    history_ids, history_mask, candidate_ids, target_index = batch

    logits_per_step = model(
        history_ids,
        history_mask,
        candidate_ids,
    )

    losses = [
        F.cross_entropy(logits, target_index).item()
        for logits in logits_per_step
    ]

    final_logits = logits_per_step[-1]
    ranks = (
        (
            torch.argsort(final_logits, dim=1, descending=True)
            == target_index.unsqueeze(1)
        )
        .nonzero(as_tuple=True)[1]
        + 1
    )

    print("One-batch diagnostic")
    print(f"Step losses: {[round(x, 4) for x in losses]}")
    print(f"Mean loss: {np.mean(losses):.4f}")
    print(f"Mean target rank: {ranks.float().mean().item():.2f}")
    print(f"HR@1: {(ranks <= 1).float().mean().item():.4f}")
    print(f"HR@10: {(ranks <= 10).float().mean().item():.4f}")
    print(f"Uniform 100-way CE: {math.log(100):.4f}")

## 18. Paths
#
Update these paths for your dataset.


In [24]:
INTERACTION_PATH = "/kaggle/input/datasets/chrisolande2/recsys/data/Luxury_Beauty_5.txt"
SBERT_EMBEDDING_PATH = "/kaggle/input/datasets/chrisolande2/recsys/data/sbert_item_embeddings.pt"

## 19. Load Data


In [25]:
user_sequences = load_user_sequences(INTERACTION_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
item_embeddings = torch.load(SBERT_EMBEDDING_PATH, map_location=device)

num_items = item_embeddings.size(0)
validate_item_indexing(user_sequences, num_items)

print(f"Users: {len(user_sequences)}")
print(f"Items: {num_items}")
print(f"Interactions: {sum(len(s) for s in user_sequences.values())}")
print(f"SBERT shape: {tuple(item_embeddings.shape)}")


Users: 3565
Items: 1473
Interactions: 32472
SBERT shape: (1473, 384)


## 20. Construct Leave-One-Out Task


In [26]:
train_pairs, val_pairs = make_train_val_pairs(user_sequences)

print(f"Train instances: {len(train_pairs)}")
print(f"Val instances: {len(val_pairs)}")


Train instances: 25342
Val instances: 3563


## 21. Lightning DataModule


In [27]:
datamodule = RecRecDataModule(
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    num_items=num_items,
    config=CONFIG,
)


## 22. Model


In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = RecRecLightning(
    pretrained_sbert_embeddings=item_embeddings,
    config=CONFIG,
).to(device)

validate_paper_compliance(model, CONFIG)
print("Paper-compliance checks passed.")



Paper-compliance checks passed.


## 23. Pre-Training Diagnostic


In [29]:
datamodule.setup("fit")
inspect_one_batch(
    model,
    datamodule.train_dataloader(),
)



One-batch diagnostic
Step losses: [4.6358, 4.7499, 4.8938, 5.074, 5.3122, 5.6022, 5.9782]
Mean loss: 5.1780
Mean target rank: 50.84
HR@1: 0.0117
HR@10: 0.0859
Uniform 100-way CE: 4.6052


## 24. EMA and Trainer


In [30]:
ema_callback = EMACallback(decay=CONFIG.ema_decay)

trainer = pl.Trainer(
    max_epochs=CONFIG.max_epochs,
    accelerator="auto",
    devices="auto",
    callbacks=[ema_callback],
    enable_progress_bar=True,
)



Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## 25. Train


In [31]:
trainer.fit(
    model,
    datamodule=datamodule,
)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ RecRec │  3.7 M │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 3.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.7 M                                                                                                
Total estimated model params size (MB): 14.683                                                                     
Modules in train mode: 29                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=50` reached.


## 26. Final Validation Ranking


In [32]:
trainer.validate(
    model,
    datamodule=datamodule,
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_hr1          │    0.3881560266017914     │
│         val_hr10          │    0.7269154787063599     │
│          val_hr5          │     0.60931795835495      │
│         val_ndcg1         │    0.3881560266017914     │
│        val_ndcg10         │    0.5404382944107056     │
│         val_ndcg5         │    0.5021987557411194     │
│         val_prec1         │    0.3881560266017914     │
│        val_prec10         │    0.07269155234098434    │
│         val_prec5         │    0.12186359614133835    │
└───────────────────────────┴───────────────────────────┘

[{'val_hr1': 0.3881560266017914,
  'val_ndcg1': 0.3881560266017914,
  'val_prec1': 0.3881560266017914,
  'val_hr5': 0.60931795835495,
  'val_ndcg5': 0.5021987557411194,
  'val_prec5': 0.12186359614133835,
  'val_hr10': 0.7269154787063599,
  'val_ndcg10': 0.5404382944107056,
  'val_prec10': 0.07269155234098434}]